# ROS 2 playground (library / notes)

Scratchpad for learning `rclpy` against the running simulation before the logic moves into the
C++ nodes in `gambit_ros/`. Three things the sim pipeline needs from ROS:

1. **subscribe** to `/camera/image` (sensor_msgs/Image) and turn it into a numpy array,
2. **call** `/world/gambit_chess/set_pose` (ros_gz_interfaces/srv/SetEntityPose) to teleport,
3. **send** a `FollowJointTrajectory` goal to `/arm_controller/follow_joint_trajectory`.

Everything ROS is behind `try: import rclpy` so this notebook also runs without ROS; the
self-check only tests the pure helpers (message <-> numpy/dict conversion).

Start the sim first: `ros2 launch gambit_bringup gambit.launch.py` (or docs/gazebo.md by hand).

**TODO checklist**
- [ ] Image message -> numpy without cv_bridge (rgb8/bgr8/mono8)
- [ ] SetEntityPose / FollowJointTrajectory goals as plain dicts (and to ROS messages)
- [ ] `CameraGrabber` node: latest frame, wait-for-fresh-frame after a teleport
- [ ] `teleport()` service client (move into teleport.Ros2Backend when it works)
- [ ] `send_arm_goal()` action client with feedback/result handling
- [ ] measure: set_pose -> first image showing the change (latency), for capture.settle_s

In [ ]:
%run ./teleport.ipynb

In [ ]:
import numpy as np

try:
    import rclpy
    from rclpy.node import Node
    HAVE_ROS = True
except ImportError:          # no ROS here: helpers below still work
    rclpy, Node, HAVE_ROS = None, object, False
print("rclpy available:", HAVE_ROS)

ARM_JOINTS = ["shoulder_pan", "shoulder_lift", "elbow_flex", "wrist_flex", "wrist_roll"]
PARK = BOARD["arm"]["park"]

## Images

`sensor_msgs/Image` has `height, width, encoding, step, data`. `cv_bridge.CvBridge().imgmsg_to_cv2(msg,
"rgb8")` is the usual way; a pure-numpy version avoids the cv_bridge/numpy version issues
that often appear on fresh installs. Gazebo publishes `rgb8` (R8G8B8 in the camera SDF).

In [ ]:
def image_msg_to_numpy(msg):
    """sensor_msgs/Image (or any object with the same fields) -> (H, W, 3) RGB or (H, W) uint8."""
    # Design:
    # - Channels from the encoding (rgb8/bgr8 -> 3, mono8 -> 1); anything else -> ValueError.
    # - np.frombuffer the data as (height, step): rows can be padded (step > width * channels),
    #   so slice off the padding before reshaping to (H, W, C).
    # - bgr8 -> reverse the channel axis; return a copy (the message buffer may be reused).
    raise NotImplementedError

## Requests as dicts

Building requests as dicts keeps them testable and printable; `to_ros_*` convert at the last moment.

In [ ]:
def trajectory_goal(positions, duration_s=3.0, joints=ARM_JOINTS):
    """control_msgs/action/FollowJointTrajectory goal (one waypoint) as a dict."""
    # Design: {trajectory: {joint_names, points: [{positions, time_from_start: {sec, nanosec}}]}};
    # one position per joint (check the length); split duration_s into whole seconds + nanoseconds.
    raise NotImplementedError

def to_ros_set_pose(req):
    """SetPose -> ros_gz_interfaces.srv.SetEntityPose.Request (needs ROS)."""
    # Design: import the srv/msg lazily; entity name + type Entity.MODEL; copy position and quaternion.
    raise NotImplementedError

def to_ros_trajectory_goal(goal):
    """dict from trajectory_goal() -> FollowJointTrajectory.Goal (needs ROS)."""
    # Design: lazy imports (control_msgs, trajectory_msgs JointTrajectoryPoint, builtin_interfaces
    # Duration); copy joint names, one JointTrajectoryPoint per dict point.
    raise NotImplementedError

## Nodes (TODO, need ROS)

Remember `use_sim_time: True` so timestamps follow Gazebo's /clock. In a notebook, spin with
`rclpy.spin_once(node, timeout_sec=0.1)` in a loop instead of `rclpy.spin` (which blocks forever).

In [ ]:
class CameraGrabber(Node):
    """TODO: keep the latest /camera/image as numpy.

    __init__: super().__init__('camera_grabber'); set use_sim_time; self.create_subscription(
    Image, '/camera/image', self._cb, qos_profile_sensor_data). _cb stores image_msg_to_numpy(msg)
    and msg.header.stamp. grab_after(stamp): spin_once until a frame newer than `stamp` arrives.
    """
    def __init__(self):
        if not HAVE_ROS:
            raise RuntimeError("needs rclpy")
        raise NotImplementedError("TODO: CameraGrabber")

def teleport(node, req, world=WORLD_NAME, timeout_s=2.0):
    """TODO: cli = node.create_client(SetEntityPose, f'/world/{world}/set_pose');
    cli.wait_for_service(timeout_s); fut = cli.call_async(to_ros_set_pose(req));
    rclpy.spin_until_future_complete(node, fut, timeout_sec=timeout_s); return fut.result().success"""
    raise NotImplementedError("TODO: teleport")

def send_arm_goal(node, goal, action="/arm_controller/follow_joint_trajectory"):
    """TODO: rclpy.action.ActionClient(node, FollowJointTrajectory, action); wait_for_server;
    send_goal_async(to_ros_trajectory_goal(goal), feedback_callback=...); wait for acceptance, then
    get_result_async(); check result.error_code == 0 (SUCCESSFUL)."""
    raise NotImplementedError("TODO: send_arm_goal")

## Try it (only with ROS + the sim running; edit and run by hand)

```python
rclpy.init()
node = rclpy.create_node('playground')
teleport(node, SetPose('w_P_5', square_center('e4')))
send_arm_goal(node, trajectory_goal([0.0, 0.2, 0.3, 1.4, 0.0]))
send_arm_goal(node, trajectory_goal(PARK))
node.destroy_node(); rclpy.shutdown()
```

## Self-check (no ROS needed)

In [ ]:
# Self-check plan (write these as asserts once the functions above work):
# - fake Image message (types.SimpleNamespace) 2 x 3 rgb8 with row padding (step = 3 * 3 + 2):
#   image_msg_to_numpy returns the original array; with encoding 'bgr8' the channels are reversed
# - unsupported encoding (e.g. '16UC1') raises ValueError
# - trajectory_goal([0, 0.2, 0.3, 1.4, 0], duration_s=2.5): joint_names == ARM_JOINTS,
#   time_from_start == {'sec': 2, 'nanosec': 500000000}; wrong number of positions is rejected
# - len(PARK) == 5
# Example shape of a test:
# msg = SimpleNamespace(height=2, width=3, encoding="rgb8", step=11, data=padded.tobytes())
print("Self-check: not written yet (see plan above)")